In [1]:
import pandas as pd
import numpy as np
import sys
import joblib 

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Affichage plus lisible
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print("Imports OK.")

pandas  : 3.0.5
numpy   : 2.5.1
Imports OK.


In [2]:
try:
    df = pd.read_csv('Student_Performance.csv')
    print("Dataset chargé avec succès.")
    print(f"Dimensions initiales : {df.shape}")
except FileNotFoundError:
    print("Erreur : Le fichier 'Student_Performance.csv' est introuvable.")
    sys.exit(1)

Dataset chargé avec succès.
Dimensions initiales : (25000, 16)


In [3]:
print("\n--- Aperçu des premières lignes ---")
print(df.head())

print("\n--- Informations sur les colonnes et types ---")
df.info()

print("\n--- Valeurs manquantes par colonne ---")
print(df.isnull().sum())

print("\n--- Nombre de doublons ---")
print(df.duplicated().sum())

print("\n--- Statistiques descriptives (numériques) ---")
print(df.describe())

print("\n--- Statistiques descriptives (catégorielles) ---")
print(df.describe(include='object'))


--- Aperçu des premières lignes ---
   student_id  age  gender school_type parent_education  study_hours  attendance_percentage internet_access travel_time extra_activities study_method  math_score  science_score  english_score  \
0           1   14    male      public    post graduate          3.1                   84.3             yes     <15 min              yes        notes        42.7           55.4           57.0   
1           2   18  female      public         graduate          3.7                   87.8             yes     >60 min               no     textbook        57.6           68.8           64.8   
2           3   17  female     private    post graduate          7.9                   65.5              no     <15 min               no        notes        84.8           95.0           79.2   
3           4   16   other      public      high school          1.1                   58.1              no   15-30 min               no        notes        44.4           27.5       

C:\Users\DELL\AppData\Local\Temp\ipykernel_7200\1266647630.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.describe(include='object'))


In [4]:
print("\n--- Vérification des valeurs invalides ---")

# Âge : doit être dans [10, 25] pour des étudiants
invalid_age = df[(df['age'] < 10) | (df['age'] > 25)]
print(f"Âges hors [10, 25] : {len(invalid_age)}")

# Notes : doivent être dans [0, 100]
score_cols = ['math_score', 'science_score', 'english_score', 'overall_score']
for col in score_cols:
    n = df[(df[col] < 0) | (df[col] > 100)].shape[0]
    print(f"{col} hors [0, 100] : {n}")

# Study hours : >= 0
print(f"study_hours < 0 : {(df['study_hours'] < 0).sum()}")

# Attendance : [0, 100]
print(f"attendance hors [0, 100] : {((df['attendance_percentage'] < 0) | (df['attendance_percentage'] > 100)).sum()}")


--- Vérification des valeurs invalides ---
Âges hors [10, 25] : 0
math_score hors [0, 100] : 0
science_score hors [0, 100] : 0
english_score hors [0, 100] : 0
overall_score hors [0, 100] : 0
study_hours < 0 : 0
attendance hors [0, 100] : 0


In [5]:
print(f"\nAvant suppression des doublons : {df.shape}")

df = df.drop_duplicates().reset_index(drop=True)  # reset_index est important
print(f"Après suppression des doublons : {df.shape}")

# Suppression de student_id (identifiant inutile pour la prédiction)
if 'student_id' in df.columns:
    df = df.drop('student_id', axis=1)
    print("Colonne 'student_id' supprimée.")


Avant suppression des doublons : (25000, 16)
Après suppression des doublons : (15000, 16)
Colonne 'student_id' supprimée.


In [6]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"Colonnes numériques : {num_cols}")
print(f"Colonnes catégorielles : {cat_cols}")

for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"  {col} : NaN remplacés par médiane = {median_val}")

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  {col} : NaN remplacés par mode = {mode_val}")

print(f"\nValeurs manquantes restantes : {df.isnull().sum().sum()}")

Colonnes numériques : ['age', 'study_hours', 'attendance_percentage', 'math_score', 'science_score', 'english_score', 'overall_score']
Colonnes catégorielles : ['gender', 'school_type', 'parent_education', 'internet_access', 'travel_time', 'extra_activities', 'study_method', 'final_grade']

Valeurs manquantes restantes : 0


In [7]:
def cap_outliers(df, column, verbose=True):
    """Ramène les valeurs extrêmes aux bornes [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    n_low = (df[column] < lower).sum()
    n_high = (df[column] > upper).sum()

    df[column] = df[column].clip(lower=lower, upper=upper)  # clip = plus propre que np.where

    if verbose:
        print(f"  {column} : {n_low} valeurs < {lower:.2f}, {n_high} valeurs > {upper:.2f} → cappées")
    return df

outlier_cols = ['study_hours', 'attendance_percentage',
                'math_score', 'science_score', 'english_score', 'overall_score']

print("\n--- Capping des outliers ---")
for col in outlier_cols:
    if col in df.columns:
        df = cap_outliers(df, col)

print("\nTraitement des outliers terminé.")


--- Capping des outliers ---
  study_hours : 0 valeurs < -3.15, 0 valeurs > 11.65 → cappées
  attendance_percentage : 0 valeurs < 25.40, 0 valeurs > 124.60 → cappées
  math_score : 5 valeurs < 0.35, 0 valeurs > 127.95 → cappées
  science_score : 4 valeurs < 0.10, 0 valeurs > 128.10 → cappées
  english_score : 2 valeurs < 0.75, 0 valeurs > 127.55 → cappées
  overall_score : 0 valeurs < 3.85, 0 valeurs > 124.25 → cappées

Traitement des outliers terminé.
